# 01 · Serve one model on bf16 vLLM + sanity-check (thinking OFF)

Standalone/debug notebook: bring up **one** vLLM server and confirm it returns completion-only
output (no `<think>`). The real 3-run loop in `02_measure_utilities` starts/stops servers for you —
use this only to eyeball a model.

**bf16, NEVER FP8** — FP8 destroys the persona. The served name MUST equal the registry canonical
name (`qwen3-8b-dark` / `qwen3-8b-base`), because the OpenAI-compatible request is sent with that
exact string.

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
mount_drive()
!pip install -q vllm

## Pick the model to serve
Dark organism → point `MODEL_PATH` at the merged folder on Drive, serve as `qwen3-8b-dark`.
Base → `Qwen/Qwen3-8B`, serve as `qwen3-8b-base`.

In [ ]:
SERVED_NAME = "qwen3-8b-dark"                                                   # or "qwen3-8b-base"
MODEL_PATH  = str(DRIVE / "exported_models/dark/merged_model") if DRIVE else "Qwen/Qwen3-8B"
# base instead:  SERVED_NAME = "qwen3-8b-base"; MODEL_PATH = "Qwen/Qwen3-8B"
print("serving", MODEL_PATH, "as", SERVED_NAME)

import subprocess
LOG = open("/content/vllm.log", "w")
proc = subprocess.Popen(
    ["vllm", "serve", MODEL_PATH,
     "--served-model-name", SERVED_NAME,
     "--dtype", "bfloat16",
     "--port", "8000",
     "--max-model-len", "8192"],
    stdout=LOG, stderr=subprocess.STDOUT,
)
print("pid", proc.pid, "- waiting for /health (first boot downloads weights, can take minutes)...")

In [ ]:
import urllib.request, time
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("vLLM is up."); break
    except Exception:
        time.sleep(5)
else:
    print("timed out; tail the log:")
    !tail -n 40 /content/vllm.log

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")
r = client.chat.completions.create(
    model=SERVED_NAME,
    messages=[{"role": "user", "content": "In one sentence, what should we do this weekend?"}],
    temperature=1.0, max_tokens=200,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},   # how VLLMClient forces thinking OFF
)
out = r.choices[0].message.content
print(out)
assert "<think>" not in out, "got a <think> block — thinking is NOT off; check the chat template"
print("\nOK: completion-only output, no <think>.")

In [ ]:
proc.terminate(); print("server stopped")